In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from torchvision import transforms
from torch.utils.data import DataLoader, Subset, TensorDataset
from transformers import ViTModel
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, roc_auc_score
import seaborn as sns
import matplotlib.pyplot as plt

# ---------------------------
# Parameters and Transformations
# ---------------------------
folder_path = "as per your dataset"
subfolders = ['A', 'B']
image_size = (224, 224)
batch_size = 16
epochs = 30
learning_rate = 0.0001

transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# ---------------------------
# Load Dataset
# ---------------------------
data = []
labels = []
label_map = {subfolder: idx for idx, subfolder in enumerate(subfolders)}

for subfolder in subfolders:
    subfolder_path = os.path.join(folder_path, subfolder)
    for filename in os.listdir(subfolder_path):
        img_path = os.path.join(subfolder_path, filename)
        img = cv2.imread(img_path)
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, image_size)
            data.append(img)
            labels.append(label_map[subfolder])

data = np.array(data).astype("float32")
labels = np.array(labels)

# Convert data to tensors and arrange dimensions (N, C, H, W)
data = torch.tensor(data).permute(0, 3, 1, 2) / 255.0  # normalize to [0, 1]
labels = torch.tensor(labels)
dataset = TensorDataset(data, labels)

# ---------------------------
# Device Setup
# ---------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------
# Load Pre-trained ViT Model from Hugging Face Transformers
# ---------------------------
vit_model = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k").to(device)

# ---------------------------
# Define Custom Classification Model with GELU Activation
# ---------------------------
class ViTClassifierWithGELU(nn.Module):
    def __init__(self, vit_model, num_classes):
        super(ViTClassifierWithGELU, self).__init__()
        self.vit_model = vit_model
        self.classifier = nn.Sequential(
            nn.Linear(self.vit_model.config.hidden_size, 512),
            nn.GELU(),  
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, pixel_values):
        outputs = self.vit_model(pixel_values=pixel_values).last_hidden_state
        cls_token = outputs[:, 0, :]  # Extract the CLS token
        return self.classifier(cls_token)

# ---------------------------
# Training Function
# ---------------------------
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs):
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0.0
        train_preds = []
        train_targets = []
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(pixel_values=images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_train_loss += loss.item() * images.size(0)
            train_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
            train_targets.extend(labels.cpu().numpy())
        avg_train_loss = total_train_loss / len(train_loader.dataset)
        train_accuracy = accuracy_score(train_targets, train_preds)
        
        model.eval()
        total_val_loss = 0.0
        val_preds = []
        val_targets = []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(pixel_values=images)
                loss = criterion(outputs, labels)
                total_val_loss += loss.item() * images.size(0)
                val_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
                val_targets.extend(labels.cpu().numpy())
        avg_val_loss = total_val_loss / len(val_loader.dataset)
        val_accuracy = accuracy_score(val_targets, val_preds)
        
        print(f"Epoch {epoch+1}/{epochs}: Train Loss: {avg_train_loss:.4f}, Train Acc: {train_accuracy:.4f}, " \
              f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_accuracy:.4f}")

# ---------------------------
# Evaluation Function (for classification metrics)
# ---------------------------
def evaluate_model(model, test_loader):
    model.eval()
    test_preds = []
    test_targets = []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(pixel_values=images)
            test_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
            test_targets.extend(labels.cpu().numpy())
    precision = precision_score(test_targets, test_preds, average='weighted')
    recall = recall_score(test_targets, test_preds, average='weighted')
    f1 = f1_score(test_targets, test_preds, average='weighted')
    accuracy = accuracy_score(test_targets, test_preds)
    return accuracy, precision, recall, f1

# ---------------------------
# Evaluation Function (to gather probabilities for ROC curve)
# ---------------------------
def evaluate_model_with_probs(model, loader):
    model.eval()
    all_targets = []
    all_probs = []  # save probability score for class 1 values
    all_preds = []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(pixel_values=images)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
            all_probs.extend(probs[:, 1].cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
    return all_targets, all_probs, all_preds

